# Data Extraction experiments

This is a notebook where I try to perform a get request over the main pages of the comune.
Then I export them as markdown or common text.
Necessary cleaning

In [ ]:
import httpx

In [ ]:
urls = [
    'https://sportellotelematico.comune.santantonioabate.na.it/action:s_italia:carta.identita;elettronica',
    "https://sportellotelematico.comune.santantonioabate.na.it/action%3As_italia%3Acarta.identita%3Bmaggiorenni",
    "https://sportellotelematico.comune.santantonioabate.na.it/action%3As_italia%3Adonare.organi",
    "https://sportellotelematico.comune.santantonioabate.na.it/action%3As_italia%3Acarta.identita%3Belettronica%3Bmaggiorenni",
    "https://sportellotelematico.comune.santantonioabate.na.it/action%3As_italia%3Acarta.identita%3Belettronica%3Bminorenni",
    "https://sportellotelematico.comune.santantonioabate.na.it/action%3As_italia%3Acarta.identita%3Belettronica%3Bdelegato",
    "https://sportellotelematico.comune.santantonioabate.na.it/action%3As_italia%3Acarta.identita%3Belettronica%3Bpin.puk",
    "https://sportellotelematico.comune.santantonioabate.na.it/action%3As_italia%3Acarta.identita%3Belettronica",
    "https://sportellotelematico.comune.santantonioabate.na.it/action%3As_italia%3Acarta.identita"
]

In [ ]:
def normalize_url(url: str):
    return url.replace("%3A", ":").replace("%3B", ";")


def get_html_from_url(url: str) -> str:
    url = normalize_url(url)
    try:
        resp = httpx.get(url)
        return str(resp.content)
    except Exception as e:
        print(f"Error occurred when processing {url=}: {str(e)}")
        return ''
    
htmls = [get_html_from_url(url) for url in urls]

In [ ]:
htmls

In [ ]:
"La carta di identità elettronica (CIE) è l'evoluzione del documento di identità in versione cartacea:".lower() in ''.join(htmls).lower()

In [ ]:
response = httpx.get(urls[2].replace("%3A", ":").replace("%3B", ";"))
response.content

In [ ]:
from trafilatura import fetch_url, extract, extract_with_metadata,html2txt
from trafilatura.settings import Extractor

html = fetch_url(normalize_url(urls[2]))
html


In [ ]:
extractor = Extractor(output_format='json',formatting=True)
html = fetch_url(normalize_url(urls[2]))
#text = extract(filecontent=html, url=normalize_url(urls[1]), options=extractor, include_formatting=True,  favor_recall=True, output_format='json', with_metadata=True)\
text = extract(

    filecontent=html,

    output_format="json",   # 👈 importante

    include_formatting=True,

    include_links=True

)
text

In [ ]:
import json


text = json.loads(text)#.keys()

text.get('comments')


text.keys()

In [ ]:
ntext = extract(filecontent=html, url=normalize_url(urls[1]), options=extractor, include_formatting=True,  favor_recall=True, output_format='txt', with_metadata=False)
ntext

In [ ]:
ntext==text

In [ ]:
normalize_url(urls[1])

In [ ]:
from urllib.parse import unquote

unquote(urls[1]) == normalize_url(urls[1])

## Parsing


In [ ]:
from bs4 import BeautifulSoup
import trafilatura
from typing import Dict, Any


SEMANTIC_TAGS = {
    "headings": ["h1", "h2", "h3"],
    "paragraphs": ["p"],
    "lists": ["li"],
    "links": ["a"],
    "emphasis": ["em", "strong"]
}


def parse_ordered(html: str) -> Dict[str, Any]:

    # opzionale: fallback pulizia globale

    clean_text = trafilatura.extract(
        html,
        include_formatting=True,
        include_links=True

    )

    soup = BeautifulSoup(html, "lxml")
    blocks = []

    # tags che vogliamo leggere IN ORDINE DOM

    allowed_tags = ["h1", "h2", "h3", "p", #"ul", "ol",
                    "li", "em", "strong", "a", "section", "article"]

    for el in soup.find_all(allowed_tags, recursive=True):
        tag = el.name
        text = el.get_text(" ", strip=True)

        if not text:
            continue

        # # LISTE

        # if tag in ["ul", "ol"]:
        #     items = [li.get_text(" ", strip=True) for li in el.find_all("li")]

        #     blocks.append({
        #         "type": "list",
        #         "items": items
        #     })
        #     continue

        # HEADINGS

        if tag in ["h1", "h2", "h3"]:
            blocks.append({
                "type": "heading",
                "level": int(tag[1]),
                "text": text
            })

            continue

        # LINKS (manteniamo contesto)

        if tag == "a":
            href = el.get("href")
            blocks.append({
                "type": "link",
                "text": text,
                "url": href
            })
            continue

        # PARAGRAFI + EM + STRONG + SECTION CONTENT
        blocks.append({
            "type": "text",
            "text": text

        })

    return {
        "raw_text": clean_text,
        "blocks": blocks

    }

In [ ]:
html = fetch_url(normalize_url(urls[2]))
textHtml = extract(filecontent=html, include_links=True, url=normalize_url(urls[1]),include_formatting=True,  favor_recall=True, output_format='html', with_metadata=True)
parse_ordered(textHtml)

In [15]:
normalize_url(urls[2])

'https://sportellotelematico.comune.santantonioabate.na.it/action:s_italia:donare.organi'

In [ ]:
from bs4 import Tag
from langchain_text_splitters.html import HTMLSemanticPreservingSplitter
html = fetch_url(normalize_url(urls[2]))

# 1. Definisci la tua funzione di gestione personalizzata per il tag <a>


def handle_links(tag: Tag) -> str:
    text = tag.get_text(strip=True)
    url = tag.get('href', '')
    if url:
        # Formattazione personalizzata: invece del markdown standard [testo](url)
        return f" {text} (Link: {url}) "
    return f" {text} "

# 2. Definisci un handler per i blocchi di citazione <blockquote> se vuoi evidenziarli


def handle_blockquote(tag: Tag) -> str:
    return f"\n[NOTA IMPORTANTE: {tag.get_text(strip=True)}]\n"


splitter = HTMLSemanticPreservingSplitter(headers_to_split_on=[("h1", "Header 1"), ("h2", "Header 2")], 
                                          preserve_links=True, max_chunk_size=2048, chunk_overlap=200, separators=["\n\n", "\n", ". ", ", ", " ", ""],
                                          custom_handlers={
    "a": handle_links,
    "blockquote": handle_blockquote
}, keep_separator=False,

denylist_tags=["head", "nav", "footer", "aside", "script", "style", "form"])
splitter.split_text(html)

[Document(metadata={}, page_content="[Salta al contenuto principale](#it-main-content) [Skip to footer content](#it-skip-to-footer-content) [Accedi all'area personale](/auth-service/login) [Regione Campania](https://www.regione.campania.it/) [Comune di Sant'Antonio Abate](/auth-service/sso-municipium?backurl=) Cerca [](/cerca) Hide navigation"),
 Document(metadata={'Header 1': 'Donare gli organi'}, page_content='Condividi [Twitter](https://twitter.com/intent/tweet?text=&url=https://sportellotelematico.comune.santantonioabate.na.it/action%3As_italia%3Adonare.organi) [Facebook](https://www.facebook.com/sharer/sharer.php?u=https://sportellotelematico.comune.santantonioabate.na.it/action%3As_italia%3Adonare.organi) [Linkedin](https://www.linkedin.com/sharing/share-offsite/?url=https://sportellotelematico.comune.santantonioabate.na.it/action%3As_italia%3Adonare.organi) [Whatsapp](whatsapp://send?text=%20https://sportellotelematico.comune.santantonioabate.na.it/action%3As_italia%3Adonare.org

In [31]:
html

'<!DOCTYPE html>\n<html lang="it" dir="ltr" prefix="og: https://ogp.me/ns#">\n<head>\n  <meta charset="utf-8" />\n<script>var _paq = _paq || [];(function(){var u=(("https:" == document.location.protocol) ? "https://nginx.piwik.prod.globogis.srl/" : "https://nginx.piwik.prod.globogis.srl/");_paq.push(["setSiteId", "437"]);_paq.push(["setTrackerUrl", u+"matomo.php"]);if (!window.matomo_search_results_active) {_paq.push(["trackPageView"]);}var d=document,g=d.createElement("script"),s=d.getElementsByTagName("script")[0];g.type="text/javascript";g.defer=true;g.async=true;g.src=u+"matomo.js";s.parentNode.insertBefore(g,s);})();</script>\n<link rel="canonical" href="https://sportellotelematico.comune.santantonioabate.na.it/action%3As_italia%3Adonare.organi" />\n<meta property="og:url" content="https://sportellotelematico.comune.santantonioabate.na.it/action%3As_italia%3Adonare.organi" />\n<meta property="og:title" content="Donare gli organi | Comune di Sant&#039;Antonio Abate" />\n<meta prope

In [37]:
docs = splitter.split_text(html)

for d in docs:
    print(d.page_content)

[Salta al contenuto principale](#it-main-content) [Skip to footer content](#it-skip-to-footer-content) [Accedi all'area personale](/auth-service/login) [Regione Campania](https://www.regione.campania.it/) [Comune di Sant'Antonio Abate](/auth-service/sso-municipium?backurl=) Cerca [](/cerca) Hide navigation
Condividi [Twitter](https://twitter.com/intent/tweet?text=&url=https://sportellotelematico.comune.santantonioabate.na.it/action%3As_italia%3Adonare.organi) [Facebook](https://www.facebook.com/sharer/sharer.php?u=https://sportellotelematico.comune.santantonioabate.na.it/action%3As_italia%3Adonare.organi) [Linkedin](https://www.linkedin.com/sharing/share-offsite/?url=https://sportellotelematico.comune.santantonioabate.na.it/action%3As_italia%3Adonare.organi) [Whatsapp](whatsapp://send?text=%20https://sportellotelematico.comune.santantonioabate.na.it/action%3As_italia%3Adonare.organi) [Normativa di riferimento](http://www.indicenormativa.it/norme/procedimenti/donare.organi?istituzione=C